In [2]:
import torch
assert torch.cuda.is_available(),
print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [4]:
!pip install -q torchxrayvision pyarrow scikit-image
from google.colab import drive; drive.mount('/content/drive')
import os
if not os.path.exists('/content/cxr'):
    !git clone -q https://github.com/DKavya8/chestxray-bias-audit.git /content/cxr
else:
    !cd /content/cxr && git pull -q

BASE       = "/content/drive/MyDrive/team-RACK-bias-paper"
RESULTS    = f"{BASE}/results"
SCORES_DIR = f"{RESULTS}/densenet_scores"
DL_SCRIPT  = f"{BASE}/data/batch_download_zips.py"
os.makedirs(SCORES_DIR, exist_ok=True)
print("inference.py?", os.path.exists('/content/cxr/inference.py'))
print("download script?", os.path.exists(DL_SCRIPT))
print("BASE?", os.path.exists(BASE))

Mounted at /content/drive
inference.py? True
download script? True
BASE? True


In [5]:
import torchxrayvision as xrv
m = xrv.models.DenseNet(weights="densenet121-res224-all")
print("cache:", xrv.utils.get_cache_dir())
print("n outputs:", len(m.pathologies))
print(list(m.pathologies)); del m

If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
cache: /root/.torchxrayvision/models_data/
n outputs: 18
['Atelectasis', 'Consolidation', 'Infiltration', 'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis', 'Effusion', 'Pneumonia', 'Pleural_Thickening', 'Cardiomegaly', 'Nodule', 'Mass', 'Hernia', 'Lung Lesion', 'Fracture', 'Lung Opacity', 'Enlarged Cardiomediastinum']


In [6]:
import os, re, glob, shutil, subprocess, sys, pyarrow.parquet as pq

TARS_TO_RUN = list(range(12))
WEIGHTS = "densenet121-res224-all"
WORK = "/content/nih"; os.makedirs(WORK, exist_ok=True)

links = re.findall(r"https://nihcc\.box\.com/shared/static/\S+?\.gz", open(DL_SCRIPT).read())
assert len(links) == 12, f"expected 12 links, found {len(links)}"

for i in TARS_TO_RUN:
    ii = f"{i:02d}"; shard = f"{SCORES_DIR}/scores_densenet_all_tar{ii}.parquet"
    if os.path.exists(shard):
        print(f"[tar {ii}] already in Drive — skip (resume)"); continue
    shutil.rmtree(f"{WORK}/images", ignore_errors=True)
    tar = f"{WORK}/images_{ii}.tar.gz"
    print(f"[tar {ii}] downloading…"); subprocess.run(["wget","-q","-O",tar,links[i]], check=True)
    print(f"[tar {ii}] extracting…"); subprocess.run(["tar","-xzf",tar,"-C",WORK], check=True)
    imgs = sorted(glob.glob(f"{WORK}/images/*.png"))
    pf = f"{WORK}/paths_{ii}.txt"; open(pf,"w").write("\n".join(imgs))
    print(f"[tar {ii}] {len(imgs)} images -> inference.py")
    subprocess.run([sys.executable,"/content/cxr/inference.py",
        "--weights",WEIGHTS,"--images-file",pf,"--output",shard,
        "--device","cuda","--batch-size","64","--workers","2"], check=True)
    n = pq.read_table(shard).num_rows
    print(f"[tar {ii}] rows {n} vs images {len(imgs)} -> {'OK' if n==len(imgs) else 'MISMATCH'}")
    shutil.rmtree(f"{WORK}/images", ignore_errors=True); os.remove(tar); os.remove(pf)
    print(f"[tar {ii}] disk cleared")

[tar 00] already in Drive — skip (resume)
[tar 01] already in Drive — skip (resume)
[tar 02] already in Drive — skip (resume)
[tar 03] already in Drive — skip (resume)
[tar 04] already in Drive — skip (resume)
[tar 05] already in Drive — skip (resume)
[tar 06] already in Drive — skip (resume)
[tar 07] already in Drive — skip (resume)
[tar 08] already in Drive — skip (resume)
[tar 09] already in Drive — skip (resume)
[tar 10] already in Drive — skip (resume)
[tar 11] already in Drive — skip (resume)


In [7]:
import glob, pandas as pd, pyarrow.parquet as pq

shards = sorted(glob.glob(f"{SCORES_DIR}/scores_densenet_all_tar*.parquet"))
print(f"found {len(shards)} shards")
scores = pd.concat([pd.read_parquet(s) for s in shards], ignore_index=True)
print("total score rows:", len(scores))

print("unique Image Index:", scores["Image Index"].nunique())
dupes = scores["Image Index"].duplicated().sum()
print("duplicate Image Index:", dupes, "->", "OK" if dupes == 0 else "PROBLEM")
print("row count check:", "OK (112120)" if len(scores) == 112120 else f"CHECK: got {len(scores)}")

meta = pd.read_csv(f"{RESULTS}/metadata_clean.csv")
score_ids = set(scores["Image Index"])
meta_ids  = set(meta["image_index"])

missing = meta_ids - score_ids
extra   = score_ids - meta_ids
print("\nmetadata images with no score:", len(missing), "->", "OK" if len(missing)==0 else "PROBLEM")
print("scored but not in metadata:", len(extra), "(expected ~14 dropped age<=0/>100 rows)")

OUT = f"{RESULTS}/densenet121_all_scores.parquet"
scores.to_parquet(OUT, index=False)
print("\nwrote:", OUT)

found 12 shards
total score rows: 112120
unique Image Index: 112120
duplicate Image Index: 0 -> OK
row count check: OK (112120)

metadata images with no score: 0 -> OK
scored but not in metadata: 14 (expected ~14 dropped age<=0/>100 rows)

wrote: /content/drive/MyDrive/team-RACK-bias-paper/results/densenet121_all_scores.parquet
